# Strategy 3 — Conservative Q-Learning (Offline RL)
## Sequential MP Adjustment Policy + Cross-Strategy Comparison

**Key innovation**: Unlike Strategies 1 & 2 (which predict *risk*), Strategy 3 learns *what to do next* — a sequential policy that recommends hourly MP adjustments.

**Clinical framing**:
- State: current patient physiology (vitals, labs, ventilator settings)
- Action: MP adjustment ∈ {−5, −2, 0, +2, +5} J/min
- Reward: multi-objective clinical utility (SpO2, MAP, lung protection, survival)
- Conservative Q-Learning: penalises out-of-distribution actions (safety for offline RL)

**Contents**:
1. MDP episode construction from MIMIC-IV
2. Custom PyTorch CQL implementation with Optuna tuning
3. SHAP-IQ on Q-network for action explainability
4. Safety Filter integration
5. Doctor-facing policy output
6. Cross-strategy comparison (S1 vs S2 vs S3)

In [ ]:
import sys, os, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.utils.helpers import load_config
from src.data.dataset import build_episodes, split_episodes, episodes_to_arrays, discretise_action, calculate_reward
from src.deployment.safety_filter import SafetyFilter
from src.deployment.explainer import ExplanationGenerator
from src.evaluation.metrics import mortality_metrics

import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import shapiq

config   = load_config(os.path.join(PROJECT_ROOT, 'config', 'config.yaml'))
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED     = config['project']['random_seed']
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Project: {config['project']['name']}")
print(f"Strategy 3: Conservative Q-Learning (Offline RL)")
print(f"Device: {DEVICE}")

## 1. Load Processed Cohort & Build Feature Space

In [ ]:
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'processed_cohort.parquet')
df = pd.read_parquet(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Stays: {df['stay_id'].nunique()}")

# Feature columns actually present in the data
AVAIL_VITALS = ['heart_rate', 'mean_arterial_pressure', 'spo2', 'temperature', 'gcs']
AVAIL_LABS   = ['ph', 'pao2', 'paco2', 'pf_ratio', 'lactate', 'creatinine', 'bilirubin', 'platelet_count']
AVAIL_VENT   = ['mechanical_power', 'tidal_volume', 'respiratory_rate', 'peep', 'fio2',
                'plateau_pressure', 'driving_pressure', 'compliance', 'minute_ventilation']
AVAIL_STATIC = ['age', 'predicted_body_weight', 'weight_kg']

def avail(cols): return [c for c in cols if c in df.columns]

FEATURE_COLS = avail(AVAIL_STATIC + AVAIL_VITALS + AVAIL_LABS + AVAIL_VENT)
STATE_DIM = len(FEATURE_COLS)
N_ACTIONS = config['mdp']['num_actions']  # 5

print(f"\nState dimension: {STATE_DIM}")
print(f"Action space:    {N_ACTIONS} discrete actions")
print(f"Features:        {FEATURE_COLS}")

## 2. Build MDP Episodes from MIMIC-IV Time-Series

In [ ]:
# Ensure hour_index exists
if 'hour_index' not in df.columns:
    df['hour_index'] = df.groupby('stay_id').cumcount()

# Median-fill NaN in feature columns before episode construction
df_clean = df.copy()
for col in FEATURE_COLS:
    med = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(med if pd.notna(med) else 0)

# Build episodes using src/data/dataset.py
episodes = build_episodes(df_clean, config, FEATURE_COLS)

total_transitions = sum(len(e['transitions']) for e in episodes)
outcomes = [e['outcome'] for e in episodes]
print(f"Episodes:     {len(episodes)}")
print(f"Transitions:  {total_transitions}")
print(f"Died:         {outcomes.count('died')} ({outcomes.count('died')/len(outcomes):.1%})")
print(f"Survived:     {outcomes.count('survived')}")

In [ ]:
# Train / val / test split (patient-level stratified)
train_eps, val_eps, test_eps = split_episodes(episodes, config)

# Flatten to arrays
obs_tr, act_tr, rew_tr, next_obs_tr, term_tr = episodes_to_arrays(train_eps)
obs_va, act_va, rew_va, next_obs_va, term_va = episodes_to_arrays(val_eps)
obs_te, act_te, rew_te, next_obs_te, term_te = episodes_to_arrays(test_eps)

print(f"Train: {obs_tr.shape[0]:>5,} transitions")
print(f"Val:   {obs_va.shape[0]:>5,} transitions")
print(f"Test:  {obs_te.shape[0]:>5,} transitions")
print(f"State dim: {obs_tr.shape[1]}")

## 3. Custom Conservative Q-Learning (CQL) in PyTorch

CQL adds a conservative regulariser to standard Q-learning:

$$\mathcal{L}_{CQL} = \underbrace{\mathbb{E}[\log\sum_a \exp Q(s,a)] - \mathbb{E}[Q(s, a_{data})]}_{\text{conservative penalty}} + \underbrace{\frac{1}{2}\mathbb{E}[(Q(s,a) - \mathcal{B}^\pi Q)^2]}_{\text{TD error}}$$

This prevents the agent from over-estimating Q-values for actions not seen in the offline data — critical for patient safety.

In [ ]:
class QNetwork(nn.Module):
    """Deep Q-Network: state → Q-values for all actions."""
    def __init__(self, state_dim: int, n_actions: int,
                 hidden_sizes=(256, 256), dropout=0.2):
        super().__init__()
        layers = []
        in_dim = state_dim
        for h in hidden_sizes:
            layers += [nn.Linear(in_dim, h), nn.LayerNorm(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, n_actions))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)


class CQLAgent:
    """
    Conservative Q-Learning agent for offline RL.
    Implements the full CQL(H) objective with target network and replay buffer.
    """
    ACTION_NAMES  = {0:'large_decrease', 1:'small_decrease', 2:'maintain',
                     3:'small_increase', 4:'large_increase'}
    ACTION_DELTAS = {0:-5, 1:-2, 2:0, 3:+2, 4:+5}
    
    def __init__(self, state_dim, n_actions, lr=3e-4, gamma=0.99,
                 cql_weight=1.0, hidden=(256,256), dropout=0.2):
        self.gamma     = gamma
        self.cql_weight = cql_weight
        self.n_actions  = n_actions
        
        self.q_net     = QNetwork(state_dim, n_actions, hidden, dropout).to(DEVICE)
        self.target_net = QNetwork(state_dim, n_actions, hidden, dropout).to(DEVICE)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()
        
        self.optimizer = torch.optim.Adam(self.q_net.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=20, gamma=0.5)
    
    def _to_tensor(self, arr):
        return torch.tensor(arr, dtype=torch.float32).to(DEVICE)
    
    def compute_loss(self, obs, actions, rewards, next_obs, terminals):
        """Compute CQL + TD loss."""
        obs     = self._to_tensor(obs)
        actions = torch.tensor(actions, dtype=torch.long).to(DEVICE)
        rewards = self._to_tensor(rewards)
        next_obs = self._to_tensor(next_obs)
        terminals = self._to_tensor(terminals)
        
        # Current Q-values
        q_all    = self.q_net(obs)                   # (B, n_actions)
        q_taken  = q_all.gather(1, actions.unsqueeze(1)).squeeze(1)  # (B,)
        
        # Target Q-values (Bellman)
        with torch.no_grad():
            next_q   = self.target_net(next_obs)      # (B, n_actions)
            max_next = next_q.max(1).values           # (B,)
            target   = rewards + self.gamma * max_next * (1.0 - terminals)
        
        # TD loss
        td_loss = F.smooth_l1_loss(q_taken, target)
        
        # CQL conservative penalty:
        # Penalise: E[log sum_a exp Q(s,a)] - E[Q(s, a_data)]
        cql_penalty = (torch.logsumexp(q_all, dim=1) - q_taken).mean()
        
        total_loss = td_loss + self.cql_weight * cql_penalty
        return total_loss, td_loss.item(), cql_penalty.item()
    
    def soft_update(self, tau=0.005):
        for tp, sp in zip(self.target_net.parameters(), self.q_net.parameters()):
            tp.data.copy_(tau * sp.data + (1 - tau) * tp.data)
    
    def predict(self, state: np.ndarray) -> dict:
        """Greedy action selection."""
        self.q_net.eval()
        with torch.no_grad():
            x = self._to_tensor(np.atleast_2d(state).astype(np.float32))
            q = self.q_net(x)[0].cpu().numpy()
        action = int(q.argmax())
        exp_q  = np.exp((q - q.max()) / 5.0)
        probs  = exp_q / exp_q.sum()
        return {'action': action,
                'action_name': self.ACTION_NAMES[action],
                'delta': self.ACTION_DELTAS[action],
                'confidence': float(probs[action]),
                'q_values': q}
    
    def predict_batch_q(self, states: np.ndarray) -> np.ndarray:
        """Return Q-values for a batch of states."""
        self.q_net.eval()
        with torch.no_grad():
            x = self._to_tensor(states.astype(np.float32))
            return self.q_net(x).cpu().numpy()


def train_cql(agent: CQLAgent, obs, actions, rewards, next_obs, terminals,
              n_epochs=60, batch_size=128, verbose=True):
    """Mini-batch training loop for CQL."""
    ds = TensorDataset(
        torch.tensor(obs,      dtype=torch.float32),
        torch.tensor(actions,  dtype=torch.long),
        torch.tensor(rewards,  dtype=torch.float32),
        torch.tensor(next_obs, dtype=torch.float32),
        torch.tensor(terminals,dtype=torch.float32),
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    
    history = {'total': [], 'td': [], 'cql': []}
    for epoch in range(n_epochs):
        agent.q_net.train()
        ep_tot, ep_td, ep_cql = [], [], []
        for batch in loader:
            o, a, r, no, t = [b.to(DEVICE) for b in batch]
            loss, td, cql = agent.compute_loss(
                o.cpu().numpy(), a.cpu().numpy(), r.cpu().numpy(),
                no.cpu().numpy(), t.cpu().numpy())
            agent.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.q_net.parameters(), 1.0)
            agent.optimizer.step()
            agent.soft_update()
            ep_tot.append(loss.item()); ep_td.append(td); ep_cql.append(cql)
        agent.scheduler.step()
        history['total'].append(np.mean(ep_tot))
        history['td'].append(np.mean(ep_td))
        history['cql'].append(np.mean(ep_cql))
        if verbose and (epoch+1) % 15 == 0:
            print(f"  Epoch {epoch+1:>3}: loss={history['total'][-1]:.4f}  "
                  f"TD={history['td'][-1]:.4f}  CQL={history['cql'][-1]:.4f}")
    return history

print("CQL architecture defined.")

## 4. Optuna Hyperparameter Search for CQL

In [ ]:
def cql_objective(trial):
    """Optuna objective: minimise validation TD loss."""
    lr         = trial.suggest_float('lr',         1e-4, 1e-2, log=True)
    cql_weight = trial.suggest_float('cql_weight', 0.1,  5.0)
    dropout    = trial.suggest_float('dropout',    0.1,  0.4)
    h1         = trial.suggest_categorical('h1',   [128, 256, 512])
    h2         = trial.suggest_categorical('h2',   [128, 256])
    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256])
    n_epochs   = 30  # short for search
    
    agent_trial = CQLAgent(
        state_dim=STATE_DIM, n_actions=N_ACTIONS,
        lr=lr, cql_weight=cql_weight,
        hidden=(h1, h2), dropout=dropout
    )
    hist = train_cql(agent_trial, obs_tr, act_tr, rew_tr, next_obs_tr, term_tr,
                     n_epochs=n_epochs, batch_size=batch_size, verbose=False)
    return hist['total'][-1]  # final loss (minimise)

print("Running Optuna hyperparameter search for CQL (30 trials) ...")
study_cql = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_cql.optimize(cql_objective, n_trials=30, show_progress_bar=False)

best = study_cql.best_params
print(f"\nBest CQL params:")
for k, v in sorted(best.items()):
    print(f"  {k}: {v}")
print(f"Best loss: {study_cql.best_value:.4f}")

## 5. Train Final CQL Agent

In [ ]:
print("Training final CQL agent ...")
best_params_cql = study_cql.best_params

final_agent = CQLAgent(
    state_dim=STATE_DIM, n_actions=N_ACTIONS,
    lr=best_params_cql['lr'],
    cql_weight=best_params_cql['cql_weight'],
    hidden=(best_params_cql['h1'], best_params_cql['h2']),
    dropout=best_params_cql['dropout']
)

history = train_cql(
    final_agent,
    obs_tr, act_tr, rew_tr, next_obs_tr, term_tr,
    n_epochs=80,
    batch_size=best_params_cql['batch_size'],
    verbose=True
)
print("Training complete.")

In [ ]:
# Training loss curve
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title, color in [
    (axes[0], 'total', 'Total Loss',  'navy'),
    (axes[1], 'td',    'TD Error',    'steelblue'),
    (axes[2], 'cql',   'CQL Penalty', 'darkorange'),
]:
    ax.plot(history[key], color=color, lw=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title(f'CQL Training: {title}')
    ax.grid(True, alpha=0.3)
plt.suptitle('Strategy 3 — CQL Training Curves', fontsize=13)
plt.tight_layout()
os.makedirs(os.path.join(PROJECT_ROOT, 'notebooks', 'figures'), exist_ok=True)
fig_path = os.path.join(PROJECT_ROOT, 'notebooks', 'figures', 'strategy3_training.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

## 6. Policy Evaluation on Test Set

In [ ]:
# Action distribution: how often does CQL agree with observed clinical actions?
final_agent.q_net.eval()
with torch.no_grad():
    q_test = final_agent.predict_batch_q(obs_te)
pred_actions = q_test.argmax(axis=1)
obs_actions  = act_te

agreement_rate = (pred_actions == obs_actions).mean()
print(f"Expert agreement rate: {agreement_rate:.1%}")

# Action distribution
action_labels = ['−5 (large dec)', '−2 (small dec)', '0 (maintain)', '+2 (small inc)', '+5 (large inc)']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
clinician_counts = np.bincount(obs_actions,  minlength=N_ACTIONS)
ai_counts        = np.bincount(pred_actions, minlength=N_ACTIONS)
x = np.arange(N_ACTIONS)
w = 0.35
ax.bar(x - w/2, clinician_counts / clinician_counts.sum(), w, label='Clinician', color='steelblue', alpha=0.8)
ax.bar(x + w/2, ai_counts        / ai_counts.sum(),        w, label='CQL Agent', color='darkorange', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(action_labels, rotation=20, ha='right')
ax.set_ylabel('Proportion'); ax.set_title('Action Distribution: Clinician vs CQL')
ax.legend()

# Reward distribution: observed vs policy
obs_returns = [sum(t['reward'] for t in ep['transitions']) for ep in test_eps]
print(f"Mean return (test): {np.mean(obs_returns):.2f} ± {np.std(obs_returns):.2f}")

ax = axes[1]
ax.hist(obs_returns, bins=20, color='steelblue', alpha=0.7, label='Observed policy')
ax.axvline(np.mean(obs_returns), color='navy', ls='--',
           label=f'Mean={np.mean(obs_returns):.1f}')
ax.set_xlabel('Cumulative episode return'); ax.set_ylabel('Count')
ax.set_title('Episode Return Distribution (Test)')
ax.legend()

plt.suptitle('Strategy 3 — CQL Policy Evaluation', fontsize=13)
plt.tight_layout()
fig_path = os.path.join(PROJECT_ROOT, 'notebooks', 'figures', 'strategy3_policy_eval.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"CQL agent agreement with observed actions: {agreement_rate:.1%}")

## 7. Safety Filter Integration

The safety filter overrides the CQL recommendation when hard clinical constraints would be violated.

In [ ]:
safety_filter = SafetyFilter(config)
explainer_gen = ExplanationGenerator()

# Evaluate safety violations on test transitions
override_count = 0
safe_actions = []
alert_types  = []

for obs, act in zip(obs_te, pred_actions):
    state_dict = {col: float(obs[i]) for i, col in enumerate(FEATURE_COLS)}
    
    safe_action, alerts = safety_filter.filter(state_dict, int(act))
    safe_actions.append(safe_action)
    if safe_action != act:
        override_count += 1
        alert_types.extend(alerts)

safe_actions = np.array(safe_actions)
override_rate = override_count / len(pred_actions)

print(f"Safety override rate: {override_rate:.1%} ({override_count}/{len(pred_actions)})")
print(f"Post-safety agreement with clinician: {(safe_actions == obs_actions).mean():.1%}")

if alert_types:
    from collections import Counter
    alert_counts = Counter(a.split(':')[0].strip() for a in alert_types)
    print("\nMost common override reasons:")
    for reason, count in alert_counts.most_common(5):
        print(f"  {count:>4}x  {reason}")

## 8. SHAP-IQ on Q-Network

We compute Shapley Interaction Indices on the Q-network to show *which features drive each action recommendation*.

In [ ]:
print("Computing SHAP-IQ for CQL Q-network...")

np.random.seed(SEED)
bg_idx = np.random.choice(len(obs_tr), size=min(50, len(obs_tr)), replace=False)

# Focus on the 'maintain' action (action 2) Q-value as target
focus_action = 2  # maintain current MP

def q_value_fn(x):
    """Return Q-value for the focus action."""
    return final_agent.predict_batch_q(x)[:, focus_action]

explainer_cql = shapiq.TabularExplainer(
    model=q_value_fn,
    data=obs_tr[bg_idx],
    index='SII',
    max_order=2,
    sample_size=64,
)

# Explain a subset of test states for speed
test_subset_idx = np.random.choice(len(obs_te), size=min(30, len(obs_te)), replace=False)
ivals = explainer_cql.explain_X(obs_te[test_subset_idx], budget=128)

all_main = np.array([iv.get_n_order_values(1) for iv in ivals])
mean_main = np.abs(all_main).mean(axis=0)
shapiq_importance = pd.Series(mean_main, index=FEATURE_COLS).sort_values(ascending=False)

print("\nTop-10 features driving Q(s, maintain):")
print(shapiq_importance.head(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Main effects
ax = axes[0]
top15 = shapiq_importance.head(15)
colors = ['#E91E63' if 'mechanical_power' in n else
          '#FF9800' if 'spo2' in n or 'pf_ratio' in n else
          '#4CAF50' if 'plateau' in n or 'driving' in n else '#607D8B'
          for n in top15.index]
ax.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1])
ax.set_xlabel('Mean |Shapley Interaction Index|')
ax.set_title('SHAP-IQ: Q-network Feature Importance\n(Q-value for "maintain" action)')

# Per-action SHAP-IQ comparison
action_shapiq = {}
for action_focus in range(N_ACTIONS):
    def q_fn_a(x, a=action_focus):
        return final_agent.predict_batch_q(x)[:, a]
    exp_a = shapiq.TabularExplainer(
        model=q_fn_a, data=obs_tr[bg_idx],
        index='SII', max_order=1, sample_size=32
    )
    ivals_a = exp_a.explain_X(obs_te[test_subset_idx[:15]], budget=64)
    main_a = np.array([iv.get_n_order_values(1) for iv in ivals_a])
    action_shapiq[action_focus] = np.abs(main_a).mean(axis=0)

action_shap_df = pd.DataFrame(action_shapiq,
    index=FEATURE_COLS,
    columns=[f'Action {i}: {CQLAgent.ACTION_NAMES[i]}' for i in range(N_ACTIONS)]
)
top10_feats = shapiq_importance.head(10).index.tolist()
action_shap_sub = action_shap_df.loc[top10_feats]

ax = axes[1]
sns.heatmap(action_shap_sub, cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Mean |SII|'})
ax.set_title('Feature Importance by Action\n(which features drive each MP adjustment?)')
ax.set_ylabel('Feature'); ax.set_xlabel('Action')
plt.xticks(rotation=30, ha='right')

plt.suptitle('Strategy 3 — SHAP-IQ on CQL Q-Network', fontsize=13, y=1.02)
plt.tight_layout()
fig_path = os.path.join(PROJECT_ROOT, 'notebooks', 'figures', 'strategy3_shapiq.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

## 9. Doctor-Facing Policy Output

*"Based on N similar patients with comparable physiology, the optimal MP adjustment is X J/min, and the survival probability is Y%"*

In [ ]:
# Build patient-level kNN for similar-patient lookup
patient_states = [ep['transitions'][0]['state'] for ep in episodes]
patient_labels = [1 if ep['outcome']=='died' else 0 for ep in episodes]
patient_ids    = [ep['stay_id'] for ep in episodes]

X_pts = np.array(patient_states)
scaler_knn = StandardScaler()
X_pts_scaled = scaler_knn.fit_transform(X_pts)

knn_sim = NearestNeighbors(n_neighbors=min(11, len(X_pts)-1), metric='euclidean')
knn_sim.fit(X_pts_scaled)

ACTION_LABELS = {
    0: '↓↓ LARGE decrease (−5 J/min)',
    1: '↓  small decrease (−2 J/min)',
    2: '→  MAINTAIN current settings',
    3: '↑  small increase (+2 J/min)',
    4: '↑↑ LARGE increase (+5 J/min)',
}

def doctor_report_cql(episode_idx: int):
    ep   = episodes[episode_idx]
    t0   = ep['transitions'][0]
    state = t0['state']
    outcome = ep['outcome']
    
    # Model prediction
    rec = final_agent.predict(state)
    state_dict = {col: float(state[i]) for i, col in enumerate(FEATURE_COLS)}
    
    # Safety check
    safe_action, alerts = safety_filter.filter(state_dict, rec['action'])
    
    # Similar patients
    x_q = scaler_knn.transform(state.reshape(1, -1))
    dists, idxs = knn_sim.kneighbors(x_q)
    sim_idxs = idxs[0][1:]  # exclude self
    sim_labels = [patient_labels[i] for i in sim_idxs]
    sim_surv_rate = np.mean([1 - l for l in sim_labels])
    n_sim = len(sim_idxs)
    
    # Optimal MP from surviving similar patients
    surv_sim = [i for i, l in zip(sim_idxs, sim_labels) if l == 0]
    if surv_sim and 'mechanical_power' in FEATURE_COLS:
        mp_idx = FEATURE_COLS.index('mechanical_power')
        opt_mp = np.median([X_pts[i, mp_idx] for i in surv_sim])
    else:
        opt_mp = None
    
    cur_mp  = state_dict.get('mechanical_power', np.nan)
    cur_spo2 = state_dict.get('spo2', np.nan)
    cur_pp   = state_dict.get('plateau_pressure', np.nan)
    cur_dp   = state_dict.get('driving_pressure', np.nan)
    
    print("=" * 70)
    print("  CLINICAL AI RECOMMENDATION — MP PERSONALISATION (Strategy 3)")
    print("=" * 70)
    print(f"  Patient:  Stay #{ep['stay_id']}  |  True outcome: {outcome.upper()}")
    print()
    print("  Current Patient State:")
    print(f"    Mechanical Power:  {cur_mp:.1f} J/min" if pd.notna(cur_mp) else "    Mechanical Power: N/A")
    print(f"    SpO2:              {cur_spo2:.0f}%" if pd.notna(cur_spo2) else "    SpO2: N/A")
    print(f"    Plateau Pressure:  {cur_pp:.0f} cmH2O" if pd.notna(cur_pp) else "    Plateau Pressure: N/A")
    print(f"    Driving Pressure:  {cur_dp:.0f} cmH2O" if pd.notna(cur_dp) else "    Driving Pressure: N/A")
    print()
    print(f"  Similar patients in DB: {n_sim}")
    print(f"  Survival rate (similar cohort): {sim_surv_rate:.0%}")
    if opt_mp is not None and pd.notna(cur_mp):
        diff = opt_mp - cur_mp
        dir_s = 'decrease' if diff < 0 else 'increase'
        print(f"  Optimal MP (from {len(surv_sim)} similar survivors): {opt_mp:.1f} J/min")
        print(f"  → Recommended: {dir_s} by {abs(diff):.1f} J/min")
    print()
    print("  AI Recommendation:")
    print(f"    Proposed action:  {ACTION_LABELS[rec['action']]}")
    if safe_action != rec['action']:
        print(f"    Safety override → {ACTION_LABELS[safe_action]}")
        for alert in alerts:
            print(f"    ⚠  {alert}")
    else:
        print(f"    Safety check:   PASSED ✓")
    print(f"    Confidence:     {rec['confidence']:.0%}")
    print()
    print("  Q-values (model's value estimate per action):")
    for i, qv in enumerate(rec['q_values']):
        marker = '← chosen' if i == rec['action'] else ''
        print(f"    {ACTION_LABELS[i]:35s} Q={qv:+6.2f}  {marker}")
    print()
    # Natural language explanation
    explanation = explainer_gen.generate(
        patient_state=state_dict,
        action=safe_action,
        confidence=rec['confidence'],
        q_values=rec['q_values'],
        similar_outcomes={'n_similar': n_sim, 'survival_rate_if_followed': sim_surv_rate}
    )
    print("  Clinical Reasoning:")
    for sentence in explanation.split('.'):
        if sentence.strip():
            print(f"    {sentence.strip()}.")
    print("=" * 70)

print("\n  === DOCTOR REPORTS — CQL POLICY RECOMMENDATIONS ===\n")
# Show 1 non-survivor and 2 survivor episodes
non_surv = [i for i, ep in enumerate(test_eps) if ep['outcome'] == 'died']
survived  = [i for i, ep in enumerate(test_eps) if ep['outcome'] == 'survived']

for demo_ep_idx in non_surv[:1] + survived[:2]:
    # Map back to global episodes index
    global_ep = episodes.index(test_eps[demo_ep_idx]) if test_eps[demo_ep_idx] in episodes else demo_ep_idx
    try:
        doctor_report_cql(global_ep)
        print()
    except Exception as e:
        print(f"  (Skipped: {e})")

## 10. Cross-Strategy Comparison: S1 vs S2 vs S3

In [ ]:
# Load saved S1 and S2 models for comparison
s1_path = os.path.join(PROJECT_ROOT, 'notebooks', 'models', 'strategy1_xgb.json')
s2_path = os.path.join(PROJECT_ROOT, 'notebooks', 'models', 'strategy2_xgb.json')

# Build the patient-level features (same as Notebooks 01 & 02) for comparison
# S1 features
AVAIL_BASE = ['age','predicted_body_weight','weight_kg','height_cm',
              'heart_rate','mean_arterial_pressure','spo2','temperature','gcs',
              'ph','pao2','paco2','pf_ratio','lactate','creatinine','bilirubin','platelet_count',
              'tidal_volume','respiratory_rate','peep','fio2','plateau_pressure',
              'driving_pressure','compliance','minute_ventilation','tidal_volume_per_kg']
AVAIL_BASE = [c for c in AVAIL_BASE if c in df.columns]

patient_records = []
for stay_id, grp in df.groupby('stay_id'):
    grp = grp.sort_values('hour_index').reset_index(drop=True)
    r   = grp.iloc[0]
    feat = {c: r.get(c, np.nan) for c in AVAIL_BASE}
    feat['stay_id'] = stay_id
    feat['label']   = int(r.get('hospital_expire_flag', 0))
    
    # S1 MP features
    h24 = grp[grp['hour_index'] <= 24]
    mp_s = h24['mechanical_power'].dropna()
    feat['mp_t0']    = grp.loc[grp['hour_index']==0,'mechanical_power'].values
    feat['mp_t0']    = feat['mp_t0'][0] if len(feat['mp_t0'])>0 else np.nan
    feat['mp_t24']   = mp_s.iloc[-1] if len(mp_s)>0 else np.nan
    feat['mp_mean']  = mp_s.mean()    if len(mp_s)>0 else np.nan
    feat['mp_max']   = mp_s.max()     if len(mp_s)>0 else np.nan
    feat['mp_min']   = mp_s.min()     if len(mp_s)>0 else np.nan
    feat['mp_delta'] = (feat['mp_t24'] - feat['mp_t0']) if (pd.notna(feat['mp_t24']) and pd.notna(feat['mp_t0'])) else np.nan
    feat['spo2_mean'] = h24['spo2'].mean();
    feat['spo2_min']  = h24['spo2'].min() if h24['spo2'].notna().any() else np.nan
    
    # S2 snapshot features
    for h in [0, 6, 12, 18, 24]:
        sub = grp[grp['hour_index'] <= h]
        if sub.empty: sub = grp.iloc[:1]
        rh = sub.iloc[-1]
        feat[f'mp_{h}h']   = rh.get('mechanical_power', np.nan)
        feat[f'spo2_{h}h'] = rh.get('spo2', np.nan)
        feat[f'dp_{h}h']   = rh.get('driving_pressure', np.nan)
        feat[f'map_{h}h']  = rh.get('mean_arterial_pressure', np.nan)
    snaps = [feat[f'mp_{h}h'] for h in [0,6,12,18,24]]
    for i in range(4):
        v0,v1 = snaps[i],snaps[i+1]
        feat[f'mp_delta_{[0,6,12,18][i]}to{[6,12,18,24][i]}'] = (v1-v0) if (pd.notna(v0) and pd.notna(v1)) else np.nan
    feat['mp_mean_24h']     = mp_s.mean()  if len(mp_s)>0 else np.nan
    feat['mp_std_24h']      = mp_s.std()   if len(mp_s)>1 else 0.0
    feat['mp_max_24h']      = mp_s.max()   if len(mp_s)>0 else np.nan
    feat['mp_trend']        = (snaps[-1]-snaps[0]) if (pd.notna(snaps[-1]) and pd.notna(snaps[0])) else np.nan
    feat['mp_above20_frac'] = float((mp_s>20).mean()) if len(mp_s)>0 else np.nan
    feat['mp_above17_frac'] = float((mp_s>17).mean()) if len(mp_s)>0 else np.nan
    feat['spo2_min_24h']    = h24['spo2'].min()  if h24['spo2'].notna().any() else np.nan
    feat['spo2_mean_24h']   = h24['spo2'].mean() if h24['spo2'].notna().any() else np.nan
    feat['n_mp_obs']        = int(mp_s.notna().sum() if hasattr(mp_s,'notna') else len(mp_s))
    patient_records.append(feat)

pt_df_all = pd.DataFrame(patient_records)
y_all     = pt_df_all['label'].values

# Build common test split (same seed → same split as in NB01 and NB02)
from sklearn.model_selection import train_test_split as tts
all_idx = np.arange(len(y_all))
tr_idx, te_idx = tts(all_idx, test_size=0.2, stratify=y_all, random_state=42)
test_stays = pt_df_all.iloc[te_idx]['stay_id'].values

print(f"Comparison test set: {len(te_idx)} patients, {y_all[te_idx].mean():.1%} mortality")

In [ ]:
comparison_results = []

# --- Strategy 1 (if model saved) ---
if os.path.exists(s1_path):
    s1_cols_base = AVAIL_BASE + ['mp_t0','mp_t24','mp_mean','mp_max','mp_min','mp_delta','spo2_mean','spo2_min']
    s1_cols = [c for c in s1_cols_base if c in pt_df_all.columns]
    X_s1 = pt_df_all[s1_cols].copy()
    for col in X_s1.columns:
        med = X_s1[col].median(); X_s1[col] = X_s1[col].fillna(med if pd.notna(med) else 0)
    X_s1_te = X_s1.iloc[te_idx].values.astype(np.float32)
    
    m1 = xgb.XGBClassifier()
    m1.load_model(s1_path)
    # Adjust feature count if needed
    n_model = m1.n_features_in_ if hasattr(m1,'n_features_in_') else None
    if n_model and n_model != X_s1_te.shape[1]:
        # Pad or truncate
        if n_model > X_s1_te.shape[1]:
            X_s1_te = np.pad(X_s1_te, ((0,0),(0,n_model-X_s1_te.shape[1])), constant_values=0)
        else:
            X_s1_te = X_s1_te[:, :n_model]
    try:
        prob_s1 = m1.predict_proba(X_s1_te)[:,1]
        auroc_s1 = roc_auc_score(y_all[te_idx], prob_s1)
        auprc_s1 = average_precision_score(y_all[te_idx], prob_s1)
        comparison_results.append({'Strategy': 'S1: XGBoost Static', 'AUROC': auroc_s1, 'AUPRC': auprc_s1})
        print(f"S1  AUROC={auroc_s1:.4f}  AUPRC={auprc_s1:.4f}")
    except Exception as e:
        print(f"S1 evaluation error: {e}")
else:
    print("S1 model not found — run Notebook 01 first")

# --- Strategy 2 (if model saved) ---
if os.path.exists(s2_path):
    s2_snap_cols = ([f'mp_{h}h' for h in [0,6,12,18,24]] +
                    [f'spo2_{h}h' for h in [0,6,12,18,24]] +
                    [f'dp_{h}h' for h in [0,6,12,18,24]] +
                    [f'map_{h}h' for h in [0,6,12,18,24]] +
                    [f'mp_delta_{a}to{b}' for a,b in [(0,6),(6,12),(12,18),(18,24)]] +
                    ['mp_mean_24h','mp_std_24h','mp_max_24h','mp_trend',
                     'mp_above20_frac','mp_above17_frac','spo2_min_24h','spo2_mean_24h','n_mp_obs'])
    s2_cols = AVAIL_BASE + [c for c in s2_snap_cols if c in pt_df_all.columns]
    s2_cols = [c for c in s2_cols if c in pt_df_all.columns]
    X_s2 = pt_df_all[s2_cols].copy()
    for col in X_s2.columns:
        med = X_s2[col].median(); X_s2[col] = X_s2[col].fillna(med if pd.notna(med) else 0)
    X_s2_te = X_s2.iloc[te_idx].values.astype(np.float32)
    
    m2 = xgb.XGBClassifier()
    m2.load_model(s2_path)
    n_model = m2.n_features_in_ if hasattr(m2,'n_features_in_') else None
    if n_model and n_model != X_s2_te.shape[1]:
        if n_model > X_s2_te.shape[1]:
            X_s2_te = np.pad(X_s2_te, ((0,0),(0,n_model-X_s2_te.shape[1])), constant_values=0)
        else:
            X_s2_te = X_s2_te[:, :n_model]
    try:
        prob_s2 = m2.predict_proba(X_s2_te)[:,1]
        auroc_s2 = roc_auc_score(y_all[te_idx], prob_s2)
        auprc_s2 = average_precision_score(y_all[te_idx], prob_s2)
        comparison_results.append({'Strategy': 'S2: XGBoost Continuous', 'AUROC': auroc_s2, 'AUPRC': auprc_s2})
        print(f"S2  AUROC={auroc_s2:.4f}  AUPRC={auprc_s2:.4f}")
    except Exception as e:
        print(f"S2 evaluation error: {e}")
else:
    print("S2 model not found — run Notebook 02 first")

# --- Strategy 3 (CQL — agreement rate + return as proxy) ---
test_returns = [sum(t['reward'] for t in ep['transitions']) for ep in test_eps]
mean_test_return = np.mean(test_returns)
comparison_results.append({
    'Strategy': 'S3: CQL Offline RL',
    'AUROC': agreement_rate,   # action agreement as proxy
    'AUPRC': float(np.clip(mean_test_return/200 + 0.5, 0, 1))  # normalised return
})
print(f"S3  Agreement={agreement_rate:.4f}  MeanReturn={mean_test_return:.2f}")

In [ ]:
if comparison_results:
    comp_df = pd.DataFrame(comparison_results)
    print("\n" + "=" * 55)
    print("  CROSS-STRATEGY COMPARISON")
    print("=" * 55)
    print(comp_df.to_string(index=False))
    print("=" * 55)
    print("\nNotes:")
    print("  S1/S2: AUROC and AUPRC measure mortality prediction accuracy")
    print("  S3:    'AUROC' = action agreement with clinician;")
    print("         'AUPRC' = normalised cumulative episode return")
    print("         S3 is evaluated on a different axis (policy quality)")
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # AUROC comparison
    ax = axes[0]
    colors = ['#2196F3', '#4CAF50', '#E91E63']
    bars = ax.bar(comp_df['Strategy'], comp_df['AUROC'],
                  color=colors[:len(comp_df)], alpha=0.8)
    ax.set_ylabel('AUROC / Agreement Rate')
    ax.set_title('Cross-Strategy: Primary Metric')
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, comp_df['AUROC']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    plt.xticks(rotation=15, ha='right')
    ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.5)
    
    # AUPRC comparison
    ax = axes[1]
    bars2 = ax.bar(comp_df['Strategy'], comp_df['AUPRC'],
                   color=colors[:len(comp_df)], alpha=0.8)
    ax.set_ylabel('AUPRC / Normalised Return')
    ax.set_title('Cross-Strategy: Secondary Metric')
    ax.set_ylim(0, 1)
    for bar, val in zip(bars2, comp_df['AUPRC']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    plt.xticks(rotation=15, ha='right')
    
    plt.suptitle('Cross-Strategy Performance Comparison', fontsize=13)
    plt.tight_layout()
    fig_path = os.path.join(PROJECT_ROOT, 'notebooks', 'figures', 'cross_strategy_comparison.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nSaved: {fig_path}")

## 11. Summary

| Strategy | Model | What it does | Key metric |
|----------|-------|--------------|------------|
| **S1: Static** | XGBoost | Predicts mortality from MP at t=0 & t=24 | AUROC |
| **S2: Continuous** | XGBoost | Predicts mortality from 5 MP time-snapshots + trajectory | AUROC |
| **S3: CQL RL** | Deep Q-Network | Recommends MP adjustments each hour | Expert agreement + return |

**Clinical recommendation framework**:
- Use **S1** for rapid admission-time risk stratification
- Use **S2** when 24h of ventilator data is available — captures trajectory effects
- Use **S3** for ongoing hourly decision support — prescriptive rather than predictive

In [ ]:
# Save CQL model weights
os.makedirs(os.path.join(PROJECT_ROOT, 'notebooks', 'models'), exist_ok=True)
torch.save({
    'q_net_state': final_agent.q_net.state_dict(),
    'config': {
        'state_dim': STATE_DIM,
        'n_actions': N_ACTIONS,
        'h1': best_params_cql['h1'],
        'h2': best_params_cql['h2'],
        'lr': best_params_cql['lr'],
        'cql_weight': best_params_cql['cql_weight'],
    },
    'feature_cols': FEATURE_COLS,
}, os.path.join(PROJECT_ROOT, 'notebooks', 'models', 'strategy3_cql.pt'))
print("CQL model saved to notebooks/models/strategy3_cql.pt")
print("\n=== Strategy 3 Complete ===")
print(f"  CQL expert agreement: {agreement_rate:.1%}")
print(f"  Safety override rate: {override_rate:.1%}")
print(f"  Mean episode return:  {np.mean(obs_returns):.2f}")